In [1]:
!pip install solara
!pip install mesa


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.9/268.9 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.1/275.1 kB 2.1 MB/s eta 0:00:00


In [2]:
import mesa
import random
#from mesa.visualization import SolaraViz, make_space_component
#from ipywidgets import interact, IntSlider, FloatSlider
import solara
#from IPython.display import display, HTML
import pandas as pd

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


In [3]:
# ========== AGENTE ==========
class Persona(mesa.Agent):
    def __init__(self, model, tipo: int, se_ha_movido: bool) -> None:
        super().__init__(model)
        self.tipo = tipo
        self.se_ha_movido = se_ha_movido

    def incomodidad(self, pos, tipo):
        contador = 0
        vecindad = self.model.grid.get_neighbors(pos, moore=True, include_center=False)
        if tipo == 1:
            for vecino in vecindad:
                if vecino.tipo == -1: contador += 1
        elif tipo == -1:
            for vecino in vecindad:
                if vecino.tipo == 1: contador += 1
        elif tipo == 0:
            for vecino in vecindad:
                contador += abs(vecino.tipo)
        return contador

    def move(self):
        self.se_ha_movido = False
        possible_steps = []
        radio = 0
        max_iteraciones = self.model.grid.width // 2 + 1
        incomodidad_base = self.incomodidad(self.pos, self.tipo)

        if incomodidad_base > self.model.tolerance:
            while not possible_steps and radio < max_iteraciones:
                radio += 1
                vecindario = self.model.grid.get_neighborhood(
                    self.pos, moore=True, include_center=False, radius=radio
                )
                empty_steps = [step for step in vecindario if self.model.grid.is_cell_empty(step)]
                possible_steps = [step for step in empty_steps if self.incomodidad(step, self.tipo) < incomodidad_base]

        if possible_steps:
            new_position = random.choice(possible_steps)
            self.model.grid.move_agent(self, new_position)
            self.se_ha_movido = True

# ========== MODELO ==========
class Modelo_Desplazamiento(mesa.Model):
    def __init__(self, N1, N2, N3, width, height, tol, seed=None):
        super().__init__(seed=seed)
        self.num_agents = N1 + N2 + N3
        self.unsatisfied_count = 0
        self.unsatisfied_by_type = {-1: 0, 0: 0, 1: 0}     # << nuevo
        #self.agent_counts = {-1: 0, 0: 0, 1: 0}
        self.grid = mesa.space.SingleGrid(width, height, True)
        self.running = True
        self.tolerance = tol
        self.steps_to_equilibrium = 0
        self.in_equilibrium = False
        self.contact_measure = {}
        self.datacollector = mesa.DataCollector(
            model_reporters={
                "Steps_to_Equilibrium": "steps_to_equilibrium",
                "Contact_Measure": "contact_measure",
                "In_Equilibrium": "in_equilibrium",
                # insatisfechos
                "Unsatisfied_Total": lambda m: m.unsatisfied_count,
                "Unsatisfied_-1":    lambda m: m.unsatisfied_by_type[-1],
                "Unsatisfied_0":     lambda m: m.unsatisfied_by_type[0],
                "Unsatisfied_1":     lambda m: m.unsatisfied_by_type[1],
            }
        )

        for i in range(N1): Persona(model=self, tipo=1, se_ha_movido=False)
        for i in range(N2): Persona(model=self, tipo=0, se_ha_movido=False)
        for i in range(N3): Persona(model=self, tipo=-1, se_ha_movido=False)

        placed_agents = set()
        for i in range(self.num_agents):
            agent = self.agents[i]
            x, y = random.randrange(self.grid.width), random.randrange(self.grid.height)
            while (x, y) in placed_agents:
                x, y = random.randrange(self.grid.width), random.randrange(self.grid.height)
            placed_agents.add((x, y))
            self.grid.place_agent(agent, (x, y))

        self.calculate_contact_measure()

    def calculate_contact_measure(self):
        """Calcula la medida de contacto desagregada por tipo de agente"""
        contact_counts = {
            -1: {-1: 0, 0: 0, 1: 0},
             0: {-1: 0, 0: 0, 1: 0},
             1: {-1: 0, 0: 0, 1: 0},
        }
        Total_contacts = 0

        for agent in self.agents:
            neighbors = self.grid.get_neighbors(agent.pos, moore=True, include_center=False)
            for neighbor in neighbors:
                contact_counts[agent.tipo][neighbor.tipo] += 1
                Total_contacts += 1

        Total_contacts = Total_contacts / 2

        for i in [-1,0,1]: contact_counts[i][i] = contact_counts[i][i] / 2

        # Guardamos proporciones
        self.contact_measure = {}
        for agent_type in [-1, 0, 1]:
            self.contact_measure[agent_type] = {}
            for neighbor_type in [-1, 0, 1]:
                if Total_contacts > 0:
                    self.contact_measure[agent_type][neighbor_type] = (
                        contact_counts[agent_type][neighbor_type] / Total_contacts
                    )
                else:
                    self.contact_measure[agent_type][neighbor_type] = 0

    def calculate_unsatisfied(self):
        """Cuenta agentes insatisfechos por tipo."""
        counts = {-1: 0, 0: 0, 1: 0}
        for agent in self.agents:
            if agent.incomodidad(agent.pos, agent.tipo) > self.tolerance:
                counts[agent.tipo] += 1
        self.unsatisfied_by_type = counts
        self.unsatisfied_count = sum(counts.values())

    def check_equilibrium(self) -> bool:
        """Verifica si el modelo ha alcanzado el equilibrio"""
        for agent in self.agents:
            if agent.se_ha_movido:
                return False
        return True

    def step(self):
        if not self.in_equilibrium:
            self.agents.shuffle_do("move")
            self.calculate_contact_measure()
            self.calculate_unsatisfied()

            if self.check_equilibrium():
                self.in_equilibrium = True
                self.running = False
            else:
                self.steps_to_equilibrium += 1

            self.datacollector.collect(self)



In [ ]:

####Ensayo con coeficiente de segregación

def run_montecarlo(n_runs=4, N1=500, N2=500, N3=500, tol=2.0, width=40, height=40, filename="resultados.xlsx") :
    resultados = []

    for run in range(1, n_runs+1):
        model = Modelo_Desplazamiento(N1, N2, N3, width, height, tol, seed=run)

        # Ejecutar hasta equilibrio
        while model.running and not model.in_equilibrium:
            model.step()

        # Medida de contacto final
        cm = model.contact_measure
        #insatisfechos = agentes_insatisfechos(model)

        # usamos la info que ya guarda el modelo
        insatis = model.unsatisfied_by_type  # ← AQUÍ la corrección

        res = {
            "Run": run,
            "Steps_to_Equilibrium": model.steps_to_equilibrium,
            # Contactos (6 medidas independientes)
            "C(-1,-1)": cm[-1][-1],
            "C(-1,0)": cm[-1][0],
            "C(-1,1)": cm[-1][1],
            "C(0,0)": cm[0][0],
            "C(0,1)": cm[0][1],
            "C(1,1)": cm[1][1],
            # Insatisfechos
            "Insat_0":  insatis[0],
            "Insat_1":  insatis[1],
            "Insat_-1": insatis[-1],
        }

                # --- segregación promedio por tipo ---
        segregacion = {}
        for tipo in [-1, 0, 1]:
            agentes_tipo = [a for a in model.agents if a.tipo == tipo]
            if agentes_tipo:
                promedios = []
                for a in agentes_tipo:
                    vecinos = model.grid.get_neighbors(a.pos, moore=True, include_center=False)
                    if vecinos:
                        mismos = sum(1 for v in vecinos if v.tipo == a.tipo)
                        promedios.append(mismos / len(vecinos))
                segregacion[tipo] = sum(promedios) / len(promedios) if promedios else 0
            else:
                segregacion[tipo] = None  # si no hay agentes de ese tipo

        res["segregacion_-1"] = segregacion[-1]
        res["segregacion_0"]  = segregacion[0]
        res["segregacion_1"]  = segregacion[1]

        resultados.append(res)

    # Pasar a DataFrame y exportar
    df = pd.DataFrame(resultados)
    df.to_excel(filename, index=False)
    return df


In [ ]:
# === BUCLE PRINCIPAL ===Montecarlo donde corre N2 de tanto en tanto sin adicionar los archivos en uno solo
for N2 in range(0, 90, 10):   # de 0 a 1500 en pasos de 6
    N3 = (90 - N2) // 2   # división entera, ajusta si quieres decimales
    N1 = (90 - N2) // 2
    print(f"Ejecutando Montecarlo con N1={N1}, N2={N2}, N3={N3}...")

    filename = f"resultados_N2_{N2}.xlsx"
    run_montecarlo(
        n_runs=2,
        N1=N1, N2=N2, N3=N3,
        tol=2.0,
        width=10, height=10,
        filename=filename
    )
    print(f" → Archivo guardado: {filename}")


import os

# Listar solo los xlsx
xlsx_files = [f for f in os.listdir() if f.endswith(".xlsx")]
print("Archivos Excel generados:", xlsx_files)


#import os

# Listar solo los xlsx
#xlsx_files = [f for f in os.listdir() if f.endswith(".xlsx")]
#print("Archivos Excel generados:", xlsx_files)

#Esto realiza analítica y gráficas a datos de excel, entrega también todo en archivo

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ===========================
# 1. Cargar el archivo Excel
# ===========================
df = pd.read_excel("resultados_N2_50.xlsx")

# Ver las primeras filas
display(df.head())

# ===========================
# 2. Estadísticos básicos
# ===========================
print("=== Estadísticos descriptivos generales ===")
resumen = df.describe()
display(resumen)

print("\n=== Medias por variable ===")
medias = df.mean()
display(medias)

print("\n=== Desviaciones estándar por variable ===")
desv = df.std()
display(desv)
"""
print("\n=== Correlaciones entre variables ===")
correlaciones = df.corr()
display(correlaciones)
"""
# Guardar en un Excel con varias hojas
with pd.ExcelWriter("analisis_numerico.xlsx") as writer:
    resumen.to_excel(writer, sheet_name="Describe")
    medias.to_excel(writer, sheet_name="Medias")
    desv.to_excel(writer, sheet_name="Desviaciones")
    correlaciones.to_excel(writer, sheet_name="Correlaciones")

print("Archivo 'analisis_numerico.xlsx' guardado con estadísticas numéricas.")
"""
# ===========================
# 3. Gráficos (opcional)
# ===========================
# Si solo quieres análisis numérico, comenta TODO lo que está en esta sección

# Histograma de todas las columnas numéricas
df.hist(figsize=(12, 10), bins=30)
plt.suptitle("Histogramas de las variables", fontsize=16)
plt.show()

# Boxplot de todas las columnas
df.boxplot(figsize=(12, 6), rot=90)
plt.title("Boxplot de todas las variables")
plt.show()

# Matriz de correlaciones como mapa de calor
import seaborn as sns
plt.figure(figsize=(10,8))
sns.heatmap(correlaciones, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Mapa de calor de correlaciones")
plt.show()

# Histograma de todas las variables
ax = df.hist(figsize=(12, 10), bins=30)
plt.suptitle("Histogramas de las variables", fontsize=16)
plt.savefig("histogramas.png", dpi=300, bbox_inches="tight")
plt.close()

# Boxplot de todas las variables
df.boxplot(figsize=(12, 6), rot=90)
plt.title("Boxplot de todas las variables")
plt.savefig("boxplot.png", dpi=300, bbox_inches="tight")
plt.close()

# Mapa de calor de correlaciones
plt.figure(figsize=(10,8))
sns.heatmap(correlaciones, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Mapa de calor de correlaciones")
plt.savefig("correlaciones.png", dpi=300, bbox_inches="tight")
plt.close()
"""
print("Gráficos guardados como 'histogramas.png', 'boxplot.png' y 'correlaciones.png'.")

#Montecarlo modificado para poder agregar todo en un mismo excel
####
import pandas as pd
import os

# Nombre del archivo final que acumulará todos los resúmenes
output_file = "analisis_montecarlo.xlsx"

# Si ya existe de antes, lo borramos para empezar limpio
if os.path.exists(output_file):
    os.remove(output_file)

# Lista para acumular todos los resúmenes
resumenes_totales = []

# === BUCLE PRINCIPAL ===
for N2 in range(0, 90, 10):   # de 0 a 90 en pasos de 10
    N3 = (90 - N2) // 2
    N1 = (90 - N2) // 2
    print(f"➡️ Ejecutando Montecarlo con N1={N1}, N2={N2}, N3={N3}...")

    # Archivo de salida de cada corrida Monte Carlo
    filename = f"resultados_N2_{N2}.xlsx"

    # Ejecutar tu simulación (esta función ya la tienes definida aparte)
    run_montecarlo(
        n_runs=5,
        N1=N1, N2=N2, N3=N3,
        tol=2.0,
        width=10, height=10,
        filename=filename
    )
    print(f"   → Archivo guardado: {filename}")

    # ================================
    # Análisis numérico de esta corrida
    # ================================
    try:
        df = pd.read_excel(filename)

        resumen = df.describe()
        resumen["N2"] = N2   # Para saber de qué corrida vienen estos datos

        # Guardar el resumen en la lista global
        resumenes_totales.append(resumen.reset_index())

    except Exception as e:
        print(f"⚠️ No se pudo analizar {filename}: {e}")

# ======================================
# Unimos todos los resúmenes y guardamos
# ======================================
if resumenes_totales:
    resumen_final = pd.concat(resumenes_totales, ignore_index=True)
    resumen_final.to_excel(output_file, index=False)
    print(f"\n✅ Archivo final '{output_file}' generado con todos los resúmenes.")
else:
    print("⚠️ No se generaron resúmenes para acumular.")

#### El mismo anterior pero variando la tolerancia
import pandas as pd
import os

# Nombre del archivo final donde guardaremos todos los resúmenes
output_file = "analisis_montecarlo_tolerancias.xlsx"

# Si ya existe de antes, lo borramos para empezar limpio
if os.path.exists(output_file):
    os.remove(output_file)

# Lista para acumular todos los resúmenes
resumenes_totales = []

# Definimos las tolerancias a probar
tolerancias = [2.0, 3.0]

# === BUCLE PRINCIPAL ===
for tol in tolerancias:
    print(f"\n🔸 Ejecutando Monte Carlo para TOLERANCIA = {tol}\n")

    for N2 in range(0, 50, 10):   # de 0 a 90 en pasos de 10
        N3 = (90 - N2) // 2
        N1 = (90 - N2) // 2
        print(f"➡️ N1={N1}, N2={N2}, N3={N3}, Tol={tol}")

        # Nombre del archivo de resultados de esta corrida
        filename = f"resultados_N2_{N2}_Tol_{int(tol)}.xlsx"

        # Ejecutar tu simulación (esta función ya la tienes definida)
        run_montecarlo(
            n_runs=2,   # puedes ajustar el número de runs aquí
            N1=N1, N2=N2, N3=N3,
            tol=tol,
            width=10, height=10,
            filename=filename
        )
        print(f"   → Archivo guardado: {filename}")

        # ================================
        # Análisis numérico de esta corrida
        # ================================
        try:
            df = pd.read_excel(filename)

            resumen = df.describe()
            resumen["N2"] = N2
            resumen["Tolerance"] = tol   # <- clave para diferenciar las tolerancias

            resumenes_totales.append(resumen.reset_index())

        except Exception as e:
            print(f"No se pudo analizar {filename}: {e}")

# ======================================
# Unimos todos los resúmenes y guardamos
# ======================================
if resumenes_totales:
    resumen_final = pd.concat(resumenes_totales, ignore_index=True)
    resumen_final.to_excel(output_file, index=False)
    print(f"\n Archivo final '{output_file}' generado con todas las tolerancias.")
else:
    print("⚠️ No se generaron resúmenes para acumular.")